# Baseline on the Real Dataset
### This Jupyter Notebook simulates 7 baseline methods on the real data.

## Import libraries

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from skmultiflow.trees import HoeffdingAdaptiveTreeClassifier
from skmultiflow.meta import AdaptiveRandomForestClassifier

from model import NormalNN, EarlyStopping, NNClassifier

from Sequential import FSCDS_SFS, DSFSC_SFS

from utils import prepare_data

## Load data

In [2]:
x_all = np.load('./dataset/weather_data.npy')
y_all = np.load('./dataset/weather_label.npy')
concept_drifts = np.load('./dataset/weather_drift_point.npy')

In [3]:
cuda = True if torch.cuda.is_available() else False

if cuda:
    torch.cuda.set_device(0)

## Baseline(All Data)

In [4]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1)
for n in range(1, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("test segment: ", n_dataset)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_prev = scaler.fit_transform(x_all[:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_prev, x_target))

    dataset_final = range(n+1)
    feature_final = range(n_feature)

    train_ds, valid_ds, test_ds = prepare_data(n, x_all_temp, y_all, concept_drifts, dataset_final, feature_final)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=n_feature, seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_NOAA_All.pt')

        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score) 
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  2
test acc: avg 0.789259, std 0.003045
data usage(#feature, #segment, %total): (8.000000, 2.000000, 100.000000%)
----------------------------------------------------------------------------
test segment:  3
test acc: avg 0.720247, std 0.006673
data usage(#feature, #segment, %total): (8.000000, 3.000000, 100.000000%)
----------------------------------------------------------------------------
test segment:  4
test acc: avg 0.798580, std 0.002380
data usage(#feature, #segment, %total): (8.000000, 4.000000, 100.000000%)
----------------------------------------------------------------------------
test segment:  5
test acc: avg 0.788519, std 0.000494
data usage(#feature, #segment, %total): (8.000000, 5.000000, 100.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.774151, std 0.003148
overall data usage(#feature, #segment, %total): (8.000000, 3.500000, 100.000000%)


## Baseline(Current Seg.)

In [5]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1)
for n in range(1, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]

    print("test segment: ", n_dataset)

    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_useless = x_all[:concept_drifts[n-1]]
    x_prev = scaler.fit_transform(x_all[concept_drifts[n-1]:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_useless, x_prev, x_target))

    dataset_final = [n]
    feature_final = range(n_feature)

    train_ds, valid_ds, test_ds = prepare_data(n, x_all_temp, y_all, concept_drifts, dataset_final, feature_final)

    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=n_feature, seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_NOAA_Current.pt')

        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")  

print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  2
test acc: avg 0.725988, std 0.003959
data usage(#feature, #segment, %total): (8.000000, 1.000000, 50.000000%)
----------------------------------------------------------------------------
test segment:  3
test acc: avg 0.699383, std 0.000000
data usage(#feature, #segment, %total): (8.000000, 1.000000, 33.333333%)
----------------------------------------------------------------------------
test segment:  4
test acc: avg 0.705988, std 0.005971
data usage(#feature, #segment, %total): (8.000000, 1.000000, 25.000000%)
----------------------------------------------------------------------------
test segment:  5
test acc: avg 0.641049, std 0.000000
data usage(#feature, #segment, %total): (8.000000, 1.000000, 20.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.693102, std 0.002483
overall data usage(#feature, #segment, %total): (8.000000, 1.000000, 32.083333%)


## Baseline(Retrain)

In [6]:
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1)
for n in range(1, len(concept_drifts)):   
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("test segment: ", n_dataset)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_useless = x_all[:concept_drifts[n-1]]
    x_prev = scaler.fit_transform(x_all[concept_drifts[n-1]:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_useless, x_prev, x_target))

    dataset_final = [n]
    feature_final = range(n_feature)
    
    x_temp = x_all_temp[concept_drifts[n-1]:concept_drifts[n]]
    y_temp = y_all[concept_drifts[n-1]:concept_drifts[n]]

    indices = list(range(len(x_temp)))
    split1 = int(len(x_temp)*0.01)
    split2 = int(len(x_temp)*0.1)
    train_indices, valid_indices, test_indices = indices[:split1], indices[split1:split2], indices[split2:]

    x_train = np.concatenate((x_temp[train_indices], x_temp[valid_indices]))
    y_train = np.concatenate((y_temp[train_indices], y_temp[valid_indices])).reshape(-1,1)
    
    x_test = x_temp[test_indices]
    y_test = y_temp[test_indices].reshape(-1,1)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        arf = HoeffdingAdaptiveTreeClassifier(random_state=s)

        # ---------------------
        #  Model training
        # ---------------------
        for i in range(len(x_train)):
            X = x_train[i]
            y = y_train[i]
            arf.partial_fit([X], y)
            
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        n_samples = 0
        correct_cnt = 0

        for i in range(len(x_test)):
            X = x_test[i]
            y = y_test[i]
            y_pred = arf.predict([X])
            if y[0] == y_pred[0]:
                correct_cnt += 1
            n_samples += 1
        
        test_score_li.append(float(correct_cnt/n_samples))

    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  2
test acc: avg 0.680741, std 0.016914
data usage(#feature, #segment, %total): (8.000000, 1.000000, 50.000000%)
----------------------------------------------------------------------------
test segment:  3
test acc: avg 0.699383, std 0.000000
data usage(#feature, #segment, %total): (8.000000, 1.000000, 33.333333%)
----------------------------------------------------------------------------
test segment:  4
test acc: avg 0.624198, std 0.012898
data usage(#feature, #segment, %total): (8.000000, 1.000000, 25.000000%)
----------------------------------------------------------------------------
test segment:  5
test acc: avg 0.709136, std 0.036229
data usage(#feature, #segment, %total): (8.000000, 1.000000, 20.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.678364, std 0.016510
overall data usage(#feature, #segment, %total): (8.000000, 1.000000, 32.083333%)


## Baseline(Adjust)

In [7]:
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1)
for n in range(1, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("n: ", n)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_prev = scaler.fit_transform(x_all[:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_prev, x_target))

    dataset_final = range(n+1)
    feature_final = range(n_feature)
    
    x_temp = x_all_temp[concept_drifts[n-1]:concept_drifts[n]]
    y_temp = y_all[concept_drifts[n-1]:concept_drifts[n]]

    indices = list(range(len(x_temp)))
    split1 = int(len(x_temp)*0.01)
    split2 = int(len(x_temp)*0.1)
    train_indices, valid_indices, test_indices = indices[:split1], indices[split1:split2], indices[split2:]

    x_train = np.concatenate((x_all_temp[:concept_drifts[n-1]], x_temp[train_indices], x_temp[valid_indices]))
    y_train = np.concatenate((y_all[:concept_drifts[n-1]], y_temp[train_indices], y_temp[valid_indices])).reshape(-1,1)
    
    x_test = x_temp[test_indices]
    y_test = y_temp[test_indices].reshape(-1,1)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        arf = HoeffdingAdaptiveTreeClassifier(random_state=s)

        # ---------------------
        #  Model training
        # ---------------------
        for i in range(len(x_train)):
            X = x_train[i]
            y = y_train[i]
            arf.partial_fit([X], y)
            
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        n_samples = 0
        correct_cnt = 0

        for i in range(len(x_test)):
            X = x_test[i]
            y = y_test[i]
            y_pred = arf.predict([X])
            if y[0] == y_pred[0]:
                correct_cnt += 1
            n_samples += 1
        
        test_score_li.append(float(correct_cnt/n_samples))

    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

n:  1
test acc: avg 0.745000, std 0.002362
data usage(#feature, #segment, %total): (8.000000, 2.000000, 100.000000%)
----------------------------------------------------------------------------
n:  2
test acc: avg 0.754691, std 0.002911
data usage(#feature, #segment, %total): (8.000000, 3.000000, 100.000000%)
----------------------------------------------------------------------------
n:  3
test acc: avg 0.767099, std 0.013837
data usage(#feature, #segment, %total): (8.000000, 4.000000, 100.000000%)
----------------------------------------------------------------------------
n:  4
test acc: avg 0.738210, std 0.006326
data usage(#feature, #segment, %total): (8.000000, 5.000000, 100.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.751250, std 0.006359
overall data usage(#feature, #segment, %total): (8.000000, 3.500000, 100.000000%)


## Baseline(Ensemble)

In [8]:
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1)
for n in range(1, len(concept_drifts)):
    n_dataset = n+1
    n_feature = x_all.shape[1]
    
    print("n: ", n)
    
    # Set the train, valid, test data
    scaler = MinMaxScaler()
    x_prev = scaler.fit_transform(x_all[:concept_drifts[n-1]+int(num*0.1)])
    x_target = scaler.transform(x_all[concept_drifts[n-1]+int(num*0.1):concept_drifts[n]])
    x_all_temp = np.concatenate((x_prev, x_target))

    dataset_final = range(n+1)
    feature_final = range(n_feature)
    
    x_temp = x_all_temp[concept_drifts[n-1]:concept_drifts[n]]
    y_temp = y_all[concept_drifts[n-1]:concept_drifts[n]]

    indices = list(range(len(x_temp)))
    split1 = int(len(x_temp)*0.01)
    split2 = int(len(x_temp)*0.1)
    train_indices, valid_indices, test_indices = indices[:split1], indices[split1:split2], indices[split2:]

    x_train = np.concatenate((x_all_temp[:concept_drifts[n-1]], x_temp[train_indices], x_temp[valid_indices]))
    y_train = np.concatenate((y_all[:concept_drifts[n-1]], y_temp[train_indices], y_temp[valid_indices])).reshape(-1,1)
    
    x_test = x_temp[test_indices]
    y_test = y_temp[test_indices].reshape(-1,1)
    
    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):
        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        arf = AdaptiveRandomForestClassifier(random_state=s)

        # ---------------------
        #  Model training
        # ---------------------
        for i in range(len(x_train)):
            X = x_train[i]
            y = y_train[i]
            arf.partial_fit([X], y)
            
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100

        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        n_samples = 0
        correct_cnt = 0

        for i in range(len(x_test)):
            X = x_test[i]
            y = y_test[i]
            y_pred = arf.predict([X])
            if y[0] == y_pred[0]:
                correct_cnt += 1
            n_samples += 1
        
        test_score_li.append(float(correct_cnt/n_samples))

    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

n:  1
test acc: avg 0.774383, std 0.001872
data usage(#feature, #segment, %total): (8.000000, 2.000000, 100.000000%)
----------------------------------------------------------------------------
n:  2
test acc: avg 0.774691, std 0.003237
data usage(#feature, #segment, %total): (8.000000, 3.000000, 100.000000%)
----------------------------------------------------------------------------
n:  3
test acc: avg 0.796728, std 0.004363
data usage(#feature, #segment, %total): (8.000000, 4.000000, 100.000000%)
----------------------------------------------------------------------------
n:  4
test acc: avg 0.775432, std 0.002860
data usage(#feature, #segment, %total): (8.000000, 5.000000, 100.000000%)
----------------------------------------------------------------------------
overall test acc: avg 0.780309, std 0.003083
overall data usage(#feature, #segment, %total): (8.000000, 3.500000, 100.000000%)


## Baseline(FSC->DS)

In [9]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1) 
for n in range(1, len(concept_drifts)):
    
    n_dataset = n+1
    n_feature = x_all.shape[1]

    # ---------------------
    #  Define FSC->DS (sequential two-step approach)
    # ---------------------
    sequential = FSCDS_SFS(x_all, y_all, n_dataset, n_feature, concept_drifts, num)
    
    # ---------------------
    #  Data calibration and sampling
    # ---------------------
    scaler_list = [MinMaxScaler(), StandardScaler(), RobustScaler()]
    x_all_cali, y_all_cali = sequential.calibration(n, scaler_list)
    x_sample, y_sample, concept_drifts_sample = sequential.sampling(n, x_all_cali, y_all_cali)
    
    print("test segment: ", n_dataset)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):

        # ---------------------
        #  Feature Selection and Calibration(FSC)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_fs(n, x_sample, y_sample, concept_drifts_sample, 
                                                             lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Data Selection(DS)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_ds(n, x_sample, y_sample, concept_drifts_sample, 
                                                             chromo_df_pcos, lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Selected and calibrated (data segment, feature) result
        # ---------------------
        dataset_final = []
        feature_final = []
        
        for i in range(n_dataset):
            if chromo_df_pcos[i]:
                dataset_final.append(i)
                
        for j in range(n_dataset, n_dataset+n_feature*(1+len(scaler_list))):
            if chromo_df_pcos[j]:
                feature_final.append(j-n_dataset)
                
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)
        
        # ---------------------
        #  Construct new train data with Quilt result 
        # ---------------------
        train_ds, valid_ds, test_ds = prepare_data(n, x_all_cali, y_all_cali, concept_drifts, 
                                                   dataset_final, feature_final)
        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=len(feature_final), seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_NOAA_FSCDS.pt')
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  2
test acc: avg 0.790988, std 0.004491
data usage(#feature, #segment, %total): (6.400000, 2.000000, 80.000000%)
----------------------------------------------------------------------------
test segment:  3
test acc: avg 0.808889, std 0.001876
data usage(#feature, #segment, %total): (8.000000, 2.600000, 86.666667%)
----------------------------------------------------------------------------
test segment:  4
test acc: avg 0.779630, std 0.025468
data usage(#feature, #segment, %total): (6.800000, 3.400000, 72.500000%)
----------------------------------------------------------------------------
test segment:  5
test acc: avg 0.780556, std 0.005800
data usage(#feature, #segment, %total): (6.800000, 4.000000, 66.500000%)
----------------------------------------------------------------------------
overall test acc: avg 0.790015, std 0.009409
overall data usage(#feature, #segment, %total): (7.000000, 3.000000, 76.416667%)


## Baseline(DS->FSC)

In [10]:
lr = 0.001
num = int(concept_drifts[-1]/len(concept_drifts))

all_test_acc = []
all_test_std = []

all_dataset = []
all_feature = []
all_usage = []

# Incremental training and evaluation from the 2nd data segment(n=1) 
for n in range(1, len(concept_drifts)):
    
    n_dataset = n+1
    n_feature = x_all.shape[1]

    # ---------------------
    #  Define DS->FSC (sequential two-step approach)
    # ---------------------
    sequential = DSFSC_SFS(x_all, y_all, n_dataset, n_feature, concept_drifts, num)
    
    scaler = MinMaxScaler()
    x_all_scale, y_all_scale = sequential.scaling(n, scaler)
    x_sample, y_sample, concept_drifts_sample = sequential.sampling(n, x_all_scale, y_all_scale)
    
    print("test segment: ", n_dataset)

    test_score_li = []
    num_dataset_li = []
    num_feature_li = []
    usage_li = []
    
    # Repeat experiment with 5 different seeds
    for s in range(5):

        # ---------------------
        #  Data Selection(DS)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_ds(n, x_sample, y_sample, concept_drifts_sample, 
                                                             lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Data calibration and sampling
        # ---------------------
        scaler_list = [MinMaxScaler(), StandardScaler(), RobustScaler()]
        x_all_cali, y_all_cali = sequential.calibration(n, scaler_list)
        x_sample, y_sample, concept_drifts_sample = sequential.sampling(n, x_all_cali, y_all_cali)
        
        # ---------------------
        #  Feature Selection and Calibration(FSC)
        # ---------------------
        chromo_df_pcos,score_pcos = sequential.generation_fs(n, x_sample, y_sample, concept_drifts_sample, 
                                                             chromo_df_pcos, lr, n_gen=50, seed=s)
        
        # ---------------------
        #  Selected and calibrated (data segment, feature) result
        # ---------------------
        dataset_final = []
        feature_final = []
        
        for i in range(n_dataset):
            if chromo_df_pcos[i]:
                dataset_final.append(i)
                
        for j in range(n_dataset, n_dataset+n_feature*(1+len(scaler_list))):
            if chromo_df_pcos[j]:
                feature_final.append(j-n_dataset)
                
        num_dataset = len(dataset_final)
        num_feature = len(feature_final)
        usage = float(num_dataset*num_feature/(n_dataset*n_feature))*100
        
        num_dataset_li.append(num_dataset)
        num_feature_li.append(num_feature)
        usage_li.append(usage)

        # ---------------------
        #  Construct new train data with Quilt result 
        # ---------------------
        train_ds, valid_ds, test_ds = prepare_data(n, x_all_cali, y_all_cali, concept_drifts, 
                                                   dataset_final, feature_final)
        train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=128, shuffle=True)

        # ---------------------
        #  Initialize model, optimizer, and criterion
        # ---------------------
        model = NormalNN(input_features=len(feature_final), seed=s)
        model = model.cuda()
        optimizer_config = {"lr": lr}
        clf = NNClassifier(model, nn.BCELoss(reduction='mean'), optim.Adam, optimizer_config)
        
        # ---------------------
        #  Model training
        # ---------------------
        clf.fit({"train": train_loader, "val": valid_loader}, epochs=2000, earlystop_path='checkpoint_NOAA_DSFSC.pt')
        
        test_output, test_loss = clf.evaluate(test_loader)
        test_score = accuracy_score(test_output['true_y'], test_output['output'])
        test_score_li.append(test_score)
        
    print('test acc: avg %f, std %f' %(np.mean(test_score_li), np.std(test_score_li)))
    all_test_acc.append(np.mean(test_score_li))
    all_test_std.append(np.std(test_score_li))
    
    print('data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
          %(np.mean(num_feature_li), np.mean(num_dataset_li), np.mean(usage_li)))
    all_dataset.append(np.mean(num_dataset_li))
    all_feature.append(np.mean(num_feature_li))
    all_usage.append(np.mean(usage_li))
    
    print("----------------------------------------------------------------------------")
    
print('overall test acc: avg %f, std %f' %(np.mean(all_test_acc), np.mean(all_test_std)))
print('overall data usage(#feature, #segment, %%total): (%f, %f, %f%%)' 
      %(np.mean(all_feature), np.mean(all_dataset), np.mean(all_usage)))

test segment:  2
test acc: avg 0.768086, std 0.021248
data usage(#feature, #segment, %total): (5.000000, 1.200000, 41.250000%)
----------------------------------------------------------------------------
test segment:  3
test acc: avg 0.774938, std 0.042441
data usage(#feature, #segment, %total): (6.800000, 1.400000, 41.666667%)
----------------------------------------------------------------------------
test segment:  4
test acc: avg 0.776173, std 0.040606
data usage(#feature, #segment, %total): (7.600000, 2.000000, 48.125000%)
----------------------------------------------------------------------------
test segment:  5
test acc: avg 0.755864, std 0.031985
data usage(#feature, #segment, %total): (6.600000, 2.200000, 38.500000%)
----------------------------------------------------------------------------
overall test acc: avg 0.768765, std 0.034070
overall data usage(#feature, #segment, %total): (6.500000, 1.700000, 42.385417%)
